In [0]:
# Section 1 — Read Bronze ERP Customer Data

df = spark.table("bike_lakehouse.bronze.erp_cust_az12")

display(df)

In [0]:
# Section 2 — Basic Data Quality Profile

from pyspark.sql.functions import col

print("Total rows:", df.count())

for column_name in df.columns:
    print(
        column_name,
        "NULLs:",
        df.filter(col(column_name).isNull()).count()
    )

In [0]:
# Section 3 — Inspect Gender Values

display(
    df
    .groupBy("GEN")
    .count()
    .orderBy(col("count").desc())
)

In [0]:
# Section 4 — Normalize Gender Values

from pyspark.sql.functions import trim, upper, when

df_gen = (
    df
    .withColumn(
        "GEN_NORMALIZED",
        upper(trim(col("GEN")))
    )
)

display(
    df_gen
    .groupBy("GEN_NORMALIZED")
    .count()
    .orderBy(col("count").desc())
)

In [0]:
# Section 5 — Birth Date Quality Check

from pyspark.sql.functions import min, max

df.select(
    min("BDATE").alias("earliest_birth_date"),
    max("BDATE").alias("latest_birth_date")
).show()

In [0]:
# Section 6 — Inspect Extreme Birth Dates

display(
    df
    .filter(col("BDATE") >= "9990-01-01")
    .select("CID", "BDATE", "GEN")
)

In [0]:
# Section 7 — Check Future Birth Dates

from pyspark.sql.functions import current_date

future_birthdates = df.filter(
    col("BDATE") > current_date()
)

print("Future birth dates:", future_birthdates.count())

display(
    future_birthdates
    .select("CID", "BDATE", "GEN")
    .orderBy("BDATE")
)

In [0]:
# Section 8 — Check Very Old Birth Dates

display(
    df
    .filter(col("BDATE") < "1900-01-01")
    .select("CID", "BDATE", "GEN")
    .orderBy("BDATE")
)

In [0]:
# Section 9 — Clean ERP Customer Data

from pyspark.sql.functions import trim, upper, when, current_date

df_clean = (
    df
    .withColumn(
        "BDATE",
        when(
            col("BDATE") > current_date(),
            None
        ).otherwise(col("BDATE"))
    )
    .withColumn(
        "GEN",
        when(
            upper(trim(col("GEN"))) == "M",
            "Male"
        )
        .when(
            upper(trim(col("GEN"))) == "MALE",
            "Male"
        )
        .when(
            upper(trim(col("GEN"))) == "F",
            "Female"
        )
        .when(
            upper(trim(col("GEN"))) == "FEMALE",
            "Female"
        )
        .otherwise("Unknown")
    )
)

df_clean.printSchema()

In [0]:
# Section 10 — Verify ERP Customer Cleaning

print("Total rows:", df_clean.count())

print(
    "NULL BDATE:",
    df_clean.filter(col("BDATE").isNull()).count()
)

print(
    "NULL GEN:",
    df_clean.filter(col("GEN").isNull()).count()
)

display(
    df_clean
    .groupBy("GEN")
    .count()
    .orderBy(col("count").desc())
)

In [0]:
# Section 11 — Check Customer ID Uniqueness

print("Total rows:", df_clean.count())

print(
    "Duplicate CID groups:",
    df_clean
    .groupBy("CID")
    .count()
    .filter(col("count") > 1)
    .count()
)

In [0]:
# Section 12 — Rename Columns

df_silver = (
    df_clean
    .withColumnRenamed("CID", "customer_id")
    .withColumnRenamed("BDATE", "birth_date")
    .withColumnRenamed("GEN", "gender")
)

df_silver.printSchema()

In [0]:
# Section 13 — Final Silver Sanity Check

print("Final row count:", df_silver.count())

print(
    "Duplicate customer IDs:",
    df_silver
    .groupBy("customer_id")
    .count()
    .filter(col("count") > 1)
    .count()
)

print(
    "NULL customer IDs:",
    df_silver.filter(col("customer_id").isNull()).count()
)

print(
    "NULL birth dates:",
    df_silver.filter(col("birth_date").isNull()).count()
)

print(
    "NULL genders:",
    df_silver.filter(col("gender").isNull()).count()
)

display(
    df_silver
    .groupBy("gender")
    .count()
    .orderBy(col("count").desc())
)

In [0]:
# Section 14 — Write ERP Customer Silver Table

df_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("bike_lakehouse.silver.erp_customers")

print("ERP Customer Silver table created successfully.")